In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2

CI_GWAS_TSV = Path(
    "<PATH_TO_CI_GWAS_FULL_RESULTS_TSV>"
)

MAIN_SETUPS = [
    "sbp_pre_post_1to60_no_cvd",
    "dbp_pre_post_1to60_no_cvd",
    "sbp_pre_post_1to60_age5_with_statins_no_cvd",
    "dbp_pre_post_1to60_age5_with_statins_no_cvd",
]

LD_FILE = Path("<PATH_TO_EXTERNAL_GWAS_LD_SNPS_CSV>")
SBP_GWAS_FILE = Path("<PATH_TO_EXTERNAL_GWAS_SBP_TSV>")
DBP_GWAS_FILE = Path("<PATH_TO_EXTERNAL_GWAS_DBP_TSV>")

OUTPREFIX = "qq_by_phenotype"
MIN_VARIANTS_PER_PHENO = 50

PHENO_RENAME = {
    # DBP
    "DBP_pre_1_ADJ": "DBP age <50",
    "DBP_pre_2_ADJ": "DBP age 50Ä‚â€žĂ˘â‚¬ĹˇÄ‚â€ąĂ‚ÂĂ„â€šĂ‹ÂÄ‚ËĂ˘â€šÂ¬ÄąË‡Ä‚â€šĂ‚Â¬Ă„â€šĂ‹ÂÄ‚ËĂ˘â‚¬ĹˇĂ‚Â¬Ă„Ä…Ă˘â‚¬Ĺź55",
    "DBP_pre_ADJ":   "DBP age-pooled",
    # SBP
    "SBP_pre_1_ADJ": "SBP age <50",
    "SBP_pre_2_ADJ": "SBP age 50Ä‚â€žĂ˘â‚¬ĹˇÄ‚â€ąĂ‚ÂĂ„â€šĂ‹ÂÄ‚ËĂ˘â€šÂ¬ÄąË‡Ä‚â€šĂ‚Â¬Ă„â€šĂ‹ÂÄ‚ËĂ˘â‚¬ĹˇĂ‚Â¬Ă„Ä…Ă˘â‚¬Ĺź55",
    "SBP_pre_ADJ":   "SBP age-pooled",
}

def pheno_label(pheno):
    return PHENO_RENAME.get(pheno, pheno)

def ld_map_from_csv(ld_filepath):
    df = pd.read_csv(ld_filepath)
    return df.groupby("BASE_SNP")["RS_ID"].agg(set).to_dict()

def load_min_pvalue_map(gwas_filepath):
    df = pd.read_csv(gwas_filepath, sep="\t")
    df["rs_id"] = df["rs_id"].astype(str)
    df["p_value"] = pd.to_numeric(df["p_value"], errors="coerce")
    df = df.dropna(subset=["rs_id", "p_value"])
    grouped = df.groupby("rs_id", as_index=False)["p_value"].min()
    return dict(zip(grouped["rs_id"], grouped["p_value"]))

def compute_min_p_for_base_snp(base_snp, base_to_rs, set[str]], pmap, float]):
    rs_set = base_to_rs.get(base_snp)
    if not rs_set:
        return None
    best = None
    for rs in rs_set:
        pv = pmap.get(rs)
        if pv is None or not (0 < pv <= 1):
            continue
        if best is None or pv < best:
            best = pv
    return best

def subset_pvals_and_union(df,
                           base_to_rs, set[str]],
                           pmap, float]):
    base_snps = df["rsID"].astype(str).unique()
    union_rs = set()
    pvals = []

    for b in base_snps:
        rs_set = base_to_rs.get(b)
        if rs_set:
            union_rs.update(rs_set)

    for b in base_snps:
        pv = compute_min_p_for_base_snp(b, base_to_rs, pmap)
        if pv is not None:
            pvals.append(pv)

    return pvals, union_rs

def build_background_pvals(pmap, float], exclude_rs):
    return [
        pv for rs, pv in pmap.items()
        if rs not in exclude_rs and 0 < pv <= 1
    ]

def qq_expected_observed(pvals):
    p = np.asarray([x for x in pvals if 0 < x <= 1], dtype=float)
    p.sort()
    n = p.size
    if n == 0:
        return np.array([]), np.array([])
    obs = -np.log10(p)
    exp = -np.log10(np.linspace(1/(2*n), 1 - 1/(2*n), n))
    return exp, obs

def lambda_gc(pvals):
    p = np.asarray([x for x in pvals if 0 < x <= 1], dtype=float)
    if p.size == 0:
        return None
    chisq = chi2.isf(p, 1)
    return float(np.median(chisq) / chi2.isf(0.5, 1))

def plot_qq(ax, pvals, *, color, label,
            ms = 3.0, alpha= 0.8):
    exp, obs = qq_expected_observed(pvals)
    if exp.size == 0:
        return
    ax.plot(exp, obs, marker="o", linestyle="none",
            ms=ms, color=color, alpha=alpha, label=label)

cigwas = pd.read_csv(CI_GWAS_TSV, sep="\t")
cigwas = cigwas.loc[cigwas["setup"].isin(MAIN_SETUPS)].copy()

cigwas["p_fdr"] = pd.to_numeric(cigwas["p_fdr"], errors="coerce")
cigwas = cigwas.loc[cigwas["p_fdr"] < 0.05].copy()

cigwas["phenotype"] = cigwas["phenotype"].astype(str)
cigwas = cigwas.loc[cigwas["phenotype"].str.contains("_pre", case=False, regex=False)].copy()

cigwas["rsID"] = cigwas["rsID"].astype(str)
cigwas = cigwas.loc[cigwas["rsID"].str.contains("rs", case=False, regex=False)].copy()

ph_counts = cigwas.groupby("phenotype")["rsID"].nunique()
keep_phenos = ph_counts[ph_counts > MIN_VARIANTS_PER_PHENO].index
cigwas = cigwas.loc[cigwas["phenotype"].isin(keep_phenos)].copy()

print("Rows after filtering:", len(cigwas))
print("Phenotypes kept:", cigwas["phenotype"].nunique())

sbp_df = cigwas.loc[cigwas["phenotype"].str.contains("sbp", case=False)].copy()
dbp_df = cigwas.loc[cigwas["phenotype"].str.contains("dbp", case=False)].copy()

print("SBP rows:", len(sbp_df), "| DBP rows:", len(dbp_df))

base_to_rs = ld_map_from_csv(LD_FILE)
sbp_pmap = load_min_pvalue_map(SBP_GWAS_FILE)
dbp_pmap = load_min_pvalue_map(DBP_GWAS_FILE)

def qq_panel(ax, trait_name, df_trait, pmap, float]):

    subset_info = {}
    union_all = set()

    for pheno, sub in df_trait.groupby("phenotype"):
        pvals, union_rs = subset_pvals_and_union(sub, base_to_rs, pmap)
        subset_info[pheno] = (pvals, union_rs)
        union_all.update(union_rs)

    bg = build_background_pvals(pmap, union_all)
    lam_bg = lambda_gc(bg)

    ax.set_title(
        f"{trait_name} QQ (background Ä‚â€žĂ˘â‚¬ĹˇĂ„Ä…Ă‹ĹĄĂ„â€šĂ˘â‚¬ĹˇÄ‚â€šĂ‚Â»={lam_bg:.3f})"
        if lam_bg is not None else
        f"{trait_name} QQ (background Ä‚â€žĂ˘â‚¬ĹˇĂ„Ä…Ă‹ĹĄĂ„â€šĂ˘â‚¬ĹˇÄ‚â€šĂ‚Â»=NA)"
    )

    plot_qq(ax, bg, color="gray",
            label=f"background (Ä‚â€žĂ˘â‚¬ĹˇĂ„Ä…Ă‹ĹĄĂ„â€šĂ˘â‚¬ĹˇÄ‚â€šĂ‚Â»={lam_bg:.3f})" if lam_bg else "background",
            ms=2.5, alpha=0.5)

    colors = ["blue", "orange", "green", "red",
              "purple", "brown", "cyan", "magenta"]

    for i, (pheno, (pvals, _)) in enumerate(sorted(subset_info.items())):
        lam = lambda_gc(pvals)
        nice = pheno_label(pheno)
        label = f"{nice} (Ä‚â€žĂ˘â‚¬ĹˇĂ„Ä…Ă‹ĹĄĂ„â€šĂ˘â‚¬ĹˇÄ‚â€šĂ‚Â»={lam:.3f})" if lam else nice
        plot_qq(ax, pvals,
                color=colors[i % len(colors)],
                label=label,
                ms=3.0,
                alpha=0.85)

    ax.set_xlabel("Expected -log10(p)")
    ax.set_ylabel("Observed -log10(p)")
    ax.legend(loc="best", fontsize=7)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

qq_panel(ax1, "SBP_pre", sbp_df, sbp_pmap)
qq_panel(ax2, "DBP_pre", dbp_df, dbp_pmap)

plt.tight_layout()
plt.savefig(f"<PATH_TO_EXTERNAL_GWAS_PNG>", dpi=150)
plt.show()

print(f"Wrote: <PATH_TO_EXTERNAL_GWAS_PNG>")